# Student Exercises — Deep Learning for Text (Chapter 11.1-11.4)

In this notebook, you will practice the fundamental steps of Natural Language Processing (NLP) using Keras and TensorFlow.

---

## Setup — Run this first (no code to write)
Just run the next cell once to download the 50,000 IMDB movie reviews and organize them into training, validation, and testing datasets. Wait until you see `Setup complete!`

In [ ]:
import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization

# Download the data
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

# Create a validation split (20% of training data)
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
    os.makedirs(val_dir / category, exist_ok=True)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

# Create batched Dataset objects
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory("aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory("aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory("aclImdb/test", batch_size=batch_size)

# Extract only text for vocabulary adaptation later
text_only_train_ds = train_ds.map(lambda x, y: x)

print("\nSetup complete!")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  9324k      0  0:00:08  0:00:08 --:--:-- 12.6M
Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.

Setup complete!


---

## Exercise 1 — Text Vectorization Foundations

### Goal
Deep learning models cannot process raw text; they require numeric tensors. In this exercise, you will manually simulate the three steps of vectorization: **Standardization**, **Tokenization**, and **Indexing**.

### Hints
*   **Standardization:** Use `text.lower()` and `re.sub(f"[{re.escape(string.punctuation)}]", "", text)` to strip punctuation.
*   **Tokenization:** The simplest method is a standard `.split()` method.
*   **Indexing:** Initialize a `TextVectorization` layer and set `output_mode="int"`.

In [ ]:
import string
import re
from tensorflow.keras.layers import TextVectorization

test_sentence = "I write, rewrite, and still rewrite again"

# Step 1: Standardization
# TODO 1: Convert the sentence to lowercase and remove punctuation
standardized_text = ...

# Step 2: Tokenization
# TODO 2: Tokenize the text by splitting it into a list of units
tokenized_text = ...

print(f"After Tokenization: {tokenized_text}")
# TODO 3: Initialize a TextVectorization layer to output integer indices
text_vectorization = ...

# Adapt and encode
text_vectorization.adapt([test_sentence])
encoded_sentence = text_vectorization(test_sentence)
print(f"After Indexing (Final Tensor): {encoded_sentence.numpy()}")

After Tokenization: Ellipsis


AttributeError: 'ellipsis' object has no attribute 'adapt'

---

## Exercise 2 — Bag-of-Words (Bigrams + TF-IDF)

### Goal
The simplest way to process text is to discard word order and treat the text as an unordered "bag" of words. You will configure a vectorizer to use **N-grams** (bigrams) encoded with **TF-IDF** weighting, and then build a dense model to classify them.

### Hints
*   **Vectorizer:** Set `ngrams=2`, `max_tokens=20000`, and `output_mode="tf_idf"`.
*   **Dense Layer:** Use 16 units and `activation="relu"`.
*   **Dropout Layer:** Set the dropout rate to `0.5` to prevent overfitting.

In [ ]:
# 1. Configure Vectorization
# TODO 1: Initialize TextVectorization for Bigrams with TF-IDF weighting
text_vectorization_tfidf = ...

print("Adapting vocabulary...")
text_vectorization_tfidf.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(lambda x, y: (text_vectorization_tfidf(x), y), num_parallel_calls=4)
tfidf_2gram_val_ds = val_ds.map(lambda x, y: (text_vectorization_tfidf(x), y), num_parallel_calls=4)

# 2. Build the Dense Model
def get_dense_model():
    inputs = keras.Input(shape=(20000,))

    # TODO 2: Add a Dense layer with 16 units and 'relu' activation
    x = ...

    # TODO 3: Add a Dropout layer with a 0.5 rate
    x = ...

    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model_bow = get_dense_model()
model_bow.fit(tfidf_2gram_train_ds.cache(), validation_data=tfidf_2gram_val_ds.cache(), epochs=3)

---

## Exercise 3 — Sequence Modeling (Embeddings + LSTM)

### Goal
Instead of manual feature engineering (like bigrams), expose the model to raw word sequences so it can figure out features on its own. You will use **Word Embeddings** and a **Bidirectional LSTM**.

### Hints
*   **Embedding Layer:** `mask_zero=True` is crucial; it tells the RNN to skip padding zeros. Set `input_dim=20000` and `output_dim=256`.
*   **Bidirectional LSTM:** Pass the `embedded` sequence into `layers.Bidirectional(layers.LSTM(32))`.

In [ ]:
max_length = 600
max_tokens = 20000

text_vectorization_seq = TextVectorization(max_tokens=max_tokens, output_mode="int", output_sequence_length=max_length)
text_vectorization_seq.adapt(text_only_train_ds)

int_train_ds = train_ds.map(lambda x, y: (text_vectorization_seq(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(lambda x, y: (text_vectorization_seq(x), y), num_parallel_calls=4)

# Build the Sequence Model
inputs = keras.Input(shape=(None,), dtype="int64")

# TODO 1: Add an Embedding layer (input_dim=max_tokens, output_dim=256, mask_zero=True)
embedded = ...

# TODO 2: Add a Bidirectional LSTM layer with 32 units
x = ...

x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_seq = keras.Model(inputs, outputs)
model_seq.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])

model_seq.fit(int_train_ds, validation_data=int_val_ds, epochs=3)

---

## Exercise 4 — Exporting an End-to-End Inference Model

### Goal
If we want to deploy our model to the real world, it needs to handle raw text. In this exercise, you will combine your vectorizer and your trained model into a single pipeline so you can feed it raw strings directly!

### Hints
*   **Step 1:** Call your `text_vectorization_tfidf` layer like a function, passing in `string_inputs`.
*   **Step 2:** Call your trained `model_bow` like a function, passing in the `processed_inputs`.

In [ ]:
# We create an input layer that accepts raw strings for you:
string_inputs = keras.Input(shape=(1,), dtype="string")

# TODO 1: Pass the `string_inputs` through your TF-IDF vectorizer
processed_inputs = ...

# TODO 2: Pass the `processed_inputs` through your trained Bag-of-Words model
outputs = ...

# We wrap it all up into a final model for you:
inference_model = keras.Model(string_inputs, outputs)

# Let's test it on a brand new review!
sample_review = tf.constant(["This movie was completely predictable and boring."])
prediction = inference_model(sample_review)
score = prediction.numpy()[0][0]

# TODO 3: Write an if/else statement to print "Positive" if the score is >= 0.5, otherwise print "Negative"
if ... :
    print(f"Result: Positive (Score: {score:.4f})")
else:
    print(f"Result: Negative (Score: {score:.4f})")

# Transformer

## Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_hub

## Load IMDB dataset

In [ ]:
max_tokens = ####        # Vocabulary size. Try: 3000, 5000, 10000
sequence_length = ####    # Review length. Try: 80, 150, 300

embed_dim = ####           # Word vector size. Try: 32, 64, 128
num_heads = ####          # Attention heads. Try: 1, 2, 4
dense_dim = ####           # Hidden layer size inside Transformer. Try: 32, 64, 128

batch_size = ####          # Try: 32 or 64
epochs = ####               # Try: 1, 2, 3


(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_tokens
)

#Using 8000 for train and 2000 for test is suggested
x_train = x_train[:####]
y_train = y_train[:####]

x_test = x_test[:####]
y_test = y_test[:####]

x_train = keras.utils.pad_sequences(x_train, maxlen=sequence_length)
x_test = keras.utils.pad_sequences(x_test, maxlen=sequence_length)

print(x_train.shape)
print(x_test.shape)

## Build model using library Transformer

In [ ]:
inputs = keras.Input(shape=(sequence_length,), dtype="int64")

x = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=max_tokens,
    sequence_length=sequence_length,
    embedding_dim=embed_dim,
    mask_zero=True
)(inputs)

x = keras_hub.layers.TransformerEncoder(
    intermediate_dim=dense_dim,
    num_heads=num_heads,
    dropout=0.1
)(x)

x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)

## We are doing Binary Classification, which activation function do we use?
## Hint: Output neuron is 1, it gives 0 to 1 Probability.

outputs = layers.Dense(1, activation="####")(x)

model = keras.Model(inputs, outputs)


# Hint
# model.compile(
#     optimizer="rms_______",
#     loss="__________",
#     metrics=["________"]
# what kind of metrics do we use for model performance?

# For sigmoid (binary classification)
#   loss = "binary_crossentropy"
# For softmax (multi-class)
#   loss = "sparse_categorical_crossentropy"
# )


model.compile(
    optimizer="####",
    loss="####",
    metrics=["####"]
)

model.summary()

## Train

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

## Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_acc:.3f}")

# Test one review

In [ ]:
word_index = keras.datasets.imdb.get_word_index()

def encode_review(text):
    words = text. #### lower the text and split it.
    encoded = []

    for word in words:
        index = word_index.get(word)
        if index is not None and index < max_tokens:
            encoded.append(index + 3)
        else:
            encoded.append(2)

    return keras.utils.pad_sequences(
        [encoded],
        maxlen=sequence_length
    )
# Give some review to see what kind of Positive Probability do you get.
review = "####"

encoded_review = encode_review(####)
prediction = model.predict(####)

print(f"Positive probability: {float(prediction[0][0]):.3f}")